In [22]:
import pandas as pd
import rdflib
from rdflib import Graph, Literal, Namespace, RDF, URIRef, OWL
from rdflib.namespace import DC, FOAF

from owlready2 import *

In [23]:
# convert ttl to RDF/XML
ttl_path = "hi_ontology.ttl"
rdfxml_path = "hi_ontology_converted.owl"

g = rdflib.Graph()
g.parse(ttl_path, format="turtle")
g.serialize(destination=rdfxml_path, format="xml")

hi = get_ontology(rdfxml_path).load()

print("Loaded ontology:", hi.base_iri)
print("Number of classes:", len(list(hi.classes())))
print("Number of object properties:", len(list(hi.object_properties())))
print("Number of data properties:", len(list(hi.data_properties())))

Loaded ontology: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/
Number of classes: 40
Number of object properties: 24
Number of data properties: 5


Classes

In [24]:

with hi:
    # Core traceability classes
    class DataArtifact(Thing):
        pass

    class ModelOutput(DataArtifact):
        pass

    class RecommendationArtifact(ModelOutput):
        pass

    class ExplanationArtifact(DataArtifact):
        pass

    class Justification(ExplanationArtifact):
        pass

    class EvidenceItem(Thing):
        pass

    # Interaction / decision
    class DecisionEvent(hi.Interaction):
        pass

    # Risks
    class Risk(Thing):
        pass

    class PrivacyRisk(Risk):
        pass

    class FairnessRisk(Risk):
        pass

    class ReliabilityRisk(Risk):
        pass

    class SafetyRisk(Risk):
        pass

    class TransparencyRisk(Risk):
        pass

    # Mitigation measures
    class MitigationMeasure(Thing):
        pass

    class HumanReview(MitigationMeasure):
        pass

    class FallbackStrategy(MitigationMeasure):
        pass

    class BiasMitigation(MitigationMeasure):
        pass

    class UncertaintyDisplay(MitigationMeasure):
        pass

    # Human factors (contextual states)
    class UserState(hi.Context):
        pass

    class TrustLevel(UserState):
        pass

    class CognitiveLoad(UserState):
        pass

    class AttentionState(UserState):
        pass

    class OverReliance(UserState):
        pass


Object properties 

In [25]:
with hi:    
    # Scenario Decision traceability
    class hasDecisionEvent(ObjectProperty):
        domain = [hi.Scenario]
        range  = [DecisionEvent]

    # Recommendation artifacts
    class hasRecommendationArtifact(ObjectProperty):
        domain = [DecisionEvent]
        range  = [RecommendationArtifact]

    class producesRecommendationArtifact(ObjectProperty):
        domain = [hi.ArtificialAgent]
        range  = [RecommendationArtifact]

    # Explanations & evidence
    class hasExplanation(ObjectProperty):
        domain = [RecommendationArtifact]
        range  = [ExplanationArtifact]

    class usesEvidence(ObjectProperty):
        domain = [DecisionEvent]
        range  = [EvidenceItem]

    class isBasedOnData(ObjectProperty):
        domain = [EvidenceItem]
        range  = [DataArtifact]

    # Risks & mitigations (attach at Scenario level; if later you add ScenarioVariant, move there)
    class hasRisk(ObjectProperty):
        domain = [hi.Scenario]
        range  = [Risk]

    class hasMitigation(ObjectProperty):
        domain = [hi.Scenario]
        range  = [MitigationMeasure]

    # Ethical considerations: attach to specific interactions/decisions if needed
    class hasEthicalConsideration(ObjectProperty):
        domain = [hi.Interaction]   # or DecisionEvent if you prefer narrower domain
        range  = [hi.EthicalConsideration]

    # Optional linking of risks to ethical considerations (NO restriction forcing it)
    class relateToEthicalConsideration(ObjectProperty):
        domain = [Risk]
        range  = [hi.EthicalConsideration]

    # Human factors attached to the decision event (contextual)
    class hasUserState(ObjectProperty):
        domain = [DecisionEvent]
        range  = [UserState]

    # Optional: keep only if you will actually use it in instances (otherwise remove)
    class affectsTrust(ObjectProperty):
        domain = [ExplanationArtifact]
        range  = [UserState]

In [26]:
DecisionEvent.is_a.append(hasRecommendationArtifact.some(RecommendationArtifact))

Data properties

In [31]:
with hi:
    class scenarioID(DataProperty, FunctionalProperty):
        domain = [hi.Scenario]
        range  = [str]

    # confidence belongs to the recommendation OUTPUT, not the hi:Recommendation task individual
    class confidenceScore(DataProperty, FunctionalProperty):
        domain = [RecommendationArtifact]
        range  = [float]

    class riskSeverity(DataProperty):
        domain = [Risk]
        range  = [int]

    class explanationType(DataProperty):
        domain = [ExplanationArtifact]
        range  = [str]

    class decisionOutcome(DataProperty, FunctionalProperty):
        domain = [DecisionEvent]
        range  = [str]
    
    class scenarioTag(DataProperty):
            domain = [hi.DecisionEvent]
            range  = [str]
    
    class label(DataProperty, FunctionalProperty):
            domain = [Thing]
            range  = [str]

print("Number of classes:", len(list(hi.classes())))
print("Number of object properties:", len(list(hi.object_properties())))
print("Number of data properties:", len(list(hi.data_properties())))

Number of classes: 40
Number of object properties: 24
Number of data properties: 7


## Individuals

In [32]:
# Human
with hi:
    Human_User = hi.Human("Human_User")
    Human_Expert = hi.Human("Human_Expert")
    Human_Clinician = hi.Human("Human_Clinician")
    Human_Learner = hi.Human("Human_Learner")
    Researcher = hi.Human("Human_Researcher")

    # Artifical agents
    A_GazeAgent = hi.ArtificialAgent("AI_GazeAwareConversationalAgent")
    A_LLM = hi.ArtificialAgent("AI_LLM_MoralReasoner")
    A_PaleoAI = hi.ArtificialAgent("AI_PaleoReconstructionAssistant")
    A_EEG = hi.ArtificialAgent("AI_EEG_TrustCalibrationSystem")
    A_Sim = hi.ArtificialAgent("AI_PolicySimulationAgent")
    A_HealthCoach = hi.ArtificialAgent("AI_SocialHealthCoach")
    A_GameAgent = hi.ArtificialAgent("AI_TrustGameTeammate")

    #DataArtifacts
    D_Gaze = hi.DataArtifact("Data_GazeTrackingStream"); D_Gaze.label = "Gaze tracking data stream"
    D_EEG = hi.DataArtifact("Data_EEGSignals"); D_EEG.label = "EEG signals"
    D_Arch = hi.DataArtifact("Data_ArchaeologyRecords"); D_Arch.label = "Archaeology / environmental records"
    D_Moral = hi.DataArtifact("Data_MoralDilemmaSet"); D_Moral.label = "Moral dilemma dataset"
    D_Policy = hi.DataArtifact("Data_PolicySimulationParameters"); D_Policy.label = "Policy simulation parameters"
    D_Lifestyle = hi.DataArtifact("Data_LifestyleLogs"); D_Lifestyle.label = "Lifestyle / behavior logs"
    D_GameLogs = hi.DataArtifact("Data_TrustGameLogs"); D_GameLogs.label = "Game interaction logs"


In [34]:
with hi:
    # VR paper
    DE_Gaze_Interaction1 = hi.DecisionEvent("DecisionEvent_GazeInteraction1")
    DE_Gaze_Interaction1.interactingAgent = [A_GazeAgent, Human_User]

    REC_Gaze_Recommendation1 = hi.RecommendationArtifact("Recommendation_GazePersonalizedResponse")
    JUST_Gaze_Explanation1 = hi.Justification("Justification_GazeBasedPersonalization")
    EV_Gaze_Evidence1 = hi.EvidenceItem("Evidence_GazeAttentionData")

    DE_Gaze_Interaction1.hasRecommendationArtifact = [REC_Gaze_Recommendation1]
    DE_Gaze_Interaction1.usesEvidence = [EV_Gaze_Evidence1]
    EV_Gaze_Evidence1.isBasedOnData = [D_Gaze]
    REC_Gaze_Recommendation1.hasExplanation = [JUST_Gaze_Explanation1]

    US_Gaze_Attention = hi.AttentionState("UserState_HighVisualAttention")
    DE_Gaze_Interaction1.hasUserState = [US_Gaze_Attention]


  
    #LLM paper
    DE_Moral_Decision1 = hi.DecisionEvent("DecisionEvent_MoralJudgement")
    DE_Moral_Decision1.interactingAgent = [A_LLM, Human_Expert]

    REC_Moral_Output1 = hi.RecommendationArtifact("Recommendation_MoralChoice")
    JUST_Moral_Explanation1 = hi.Justification("Justification_MoralReasoning")
    EV_Moral_Evidence1 = hi.EvidenceItem("Evidence_MoralDataset")

    DE_Moral_Decision1.hasRecommendationArtifact = [REC_Moral_Output1]
    DE_Moral_Decision1.usesEvidence = [EV_Moral_Evidence1]
    EV_Moral_Evidence1.isBasedOnData = [D_Moral]
    REC_Moral_Output1.hasExplanation = [JUST_Moral_Explanation1]

    US_Moral_Trust = hi.TrustLevel("UserState_TrustInLLM")
    DE_Moral_Decision1.hasUserState = [US_Moral_Trust]


   
    # 3. Archaeology AI paper
    DE_Paleo_Analysis1 = hi.DecisionEvent("DecisionEvent_EnvironmentReconstruction")
    DE_Paleo_Analysis1.interactingAgent = [A_PaleoAI, Researcher]

    REC_Paleo_Output1 = hi.RecommendationArtifact("Recommendation_ReconstructionHypothesis")
    JUST_Paleo_Explanation1 = hi.Justification("Justification_DataDrivenReconstruction")
    EV_Paleo_Evidence1 = hi.EvidenceItem("Evidence_ArchaeologicalRecords")

    DE_Paleo_Analysis1.hasRecommendationArtifact = [REC_Paleo_Output1]
    DE_Paleo_Analysis1.usesEvidence = [EV_Paleo_Evidence1]
    EV_Paleo_Evidence1.isBasedOnData = [D_Arch]
    REC_Paleo_Output1.hasExplanation = [JUST_Paleo_Explanation1]

    US_Paleo_CognitiveLoad = hi.CognitiveLoad("UserState_ResearchCognitiveLoad")
    DE_Paleo_Analysis1.hasUserState = [US_Paleo_CognitiveLoad]


    
    # 4.EEG paper
    DE_EEG_Decision1 = hi.DecisionEvent("DecisionEvent_TrustCalibration")
    DE_EEG_Decision1.interactingAgent = [A_EEG, Human_User]

    REC_EEG_Output1 = hi.RecommendationArtifact("Recommendation_TrustAdjustment")
    JUST_EEG_Explanation1 = hi.Justification("Justification_EEGSignalAnalysis")
    EV_EEG_Evidence1 = hi.EvidenceItem("Evidence_EEGSignalsUsed")

    DE_EEG_Decision1.hasRecommendationArtifact = [REC_EEG_Output1]
    DE_EEG_Decision1.usesEvidence = [EV_EEG_Evidence1]
    EV_EEG_Evidence1.isBasedOnData = [D_EEG]
    REC_EEG_Output1.hasExplanation = [JUST_EEG_Explanation1]

    US_EEG_Trust = hi.TrustLevel("UserState_TrustCalibrationLevel")
    DE_EEG_Decision1.hasUserState = [US_EEG_Trust]


    # Organ donation simulation paper
    DE_Sim_Decision1 = hi.DecisionEvent("DecisionEvent_PolicySimulation")
    DE_Sim_Decision1.interactingAgent = [A_Sim, Human_Expert]

    REC_Sim_Output1 = hi.RecommendationArtifact("Recommendation_PolicyOption")
    JUST_Sim_Explanation1 = hi.Justification("Justification_SimulationResults")
    EV_Sim_Evidence1 = hi.EvidenceItem("Evidence_SimulationParameters")

    DE_Sim_Decision1.hasRecommendationArtifact = [REC_Sim_Output1]
    DE_Sim_Decision1.usesEvidence = [EV_Sim_Evidence1]
    EV_Sim_Evidence1.isBasedOnData = [D_Policy]
    REC_Sim_Output1.hasExplanation = [JUST_Sim_Explanation1]
    
    # 6. Health lifestyle paper
    DE_Health_Decision1 = hi.DecisionEvent("DecisionEvent_LifestyleRecommendation")
    DE_Health_Decision1.interactingAgent = [A_HealthCoach, Human_User]

    REC_Health_Output1 = hi.RecommendationArtifact("Recommendation_LifestyleChange")
    JUST_Health_Explanation1 = hi.Justification("Justification_BehavioralAnalysis")
    EV_Health_Evidence1 = hi.EvidenceItem("Evidence_LifestyleLogs")

    DE_Health_Decision1.hasRecommendationArtifact = [REC_Health_Output1]
    DE_Health_Decision1.usesEvidence = [EV_Health_Evidence1]
    EV_Health_Evidence1.isBasedOnData = [D_Lifestyle]
    REC_Health_Output1.hasExplanation = [JUST_Health_Explanation1]

    US_Health_OverReliance = hi.OverReliance("UserState_AIOverReliance")
    DE_Health_Decision1.hasUserState = [US_Health_OverReliance]

    #Trust factors game paper
    DE_Game_Decision1 = hi.DecisionEvent("DecisionEvent_CollaborativeGameStrategy")
    DE_Game_Decision1.interactingAgent = [A_GameAgent, Human_User]

    REC_Game_Output1 = hi.RecommendationArtifact("Recommendation_CollaborationStrategy")
    JUST_Game_Explanation1 = hi.Justification("Justification_TrustFactorModel")
    EV_Game_Evidence1 = hi.EvidenceItem("Evidence_GameInteractionLogs")

    DE_Game_Decision1.hasRecommendationArtifact = [REC_Game_Output1]
    DE_Game_Decision1.usesEvidence = [EV_Game_Evidence1]
    EV_Game_Evidence1.isBasedOnData = [D_GameLogs]
    REC_Game_Output1.hasExplanation = [JUST_Game_Explanation1]

    US_Game_Trust = hi.TrustLevel("UserState_TrustInAITeammate")
    DE_Game_Decision1.hasUserState = [US_Game_Trust]


print("Total individuals:", len(list(hi.individuals())))

Total individuals: 81


In [37]:
graph = hi.world.as_rdflib_graph()
graph.serialize(destination="hi_ontology_with_instances.ttl", format="turtle")

print("TTL export complete.")

TTL export complete.
